# Лабораторна 1. Основні бібліотеки. Лінійні моделі навчання.

В цій лабораторній роботі студенти:
- ознайомлюються з сервісом Colab;
- ознайомлюються з базовими бібліотеками для машинного навчання;
- тренують неглибокі лінійні моделі машинного навчання на базі простих регресій: лінійна, логістична, поліноміальна.

## 0. Налаштування середовища

### 0.0. Імпортуємо залежності

In [ ]:
import sys
import os
import random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

### 0.1. Перевіряємо чи ми точно в Colab

In [ ]:
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

### 0.2. Фіксуємо всі джерела випадковості. 
На GPU повна детермінованість трохи сповільнює тренування
(`cudnn.deterministic = True` вимикає частину швидких алгоритмів) — 
тут це не критично, адже дані й модель дрібні.

In [ ]:
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

**Вправа 1.** Скористайся вбудованим модулем `random`. Спробуй `random.randint(1, 100)` і `random.shuffle()` на списку — випадковість буває не лише в числі, а й у перестановці. Згенеруй обома способами двічі ДО виклику `set_seed()` — переконайся, що результати різні. Потім ще двічі **ПІСЛЯ** з викликом set_seed() після кожної генерації і потім ще двічі **без проміжного виклику** `set_seed()`. 

Після клітинки з кодом створи клітинку з маркдауном і поясни що відбувається після `set_seed()` і як це проявляється в результатах виконання коду.

In [ ]:
def generate():
    numbers = list(range(10))
    number = random.randint(1, 100)
    random.shuffle(numbers)
    print(number, numbers)
print('До set_seed:')
for _ in range(2): generate()
print('З повторним set_seed:')
for _ in range(2):
    set_seed()
    generate()
print('Без повторного set_seed:')
for _ in range(2): generate()

**Пояснення.** Seed задає початковий стан генератора псевдовипадкових чисел. Повторне встановлення seed відтворює ту саму послідовність чисел і перестановок. Без повторного встановлення стан генератора змінюється, тому результати зазвичай різні.

### 0.3. Використовуємо лише CPU (в цій лабі)
На Colab зазвичай є безкоштовний GPU (T4), але для лінійної регресії
на кількасот точок CPU часто не повільніший — виграш GPU з'їдає пересилання
даних на пристрій і назад щоепохи. Тому нижче свідомо тренуємось на CPU.

In [ ]:
device_available = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

print("Доступний пристрій :", device_available)
print("Використовуємо     :", device)
if device_available.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

assert device.type == "cpu", f"Ця лаба має виконуватися на CPU, а не на GPU ({device})"

**Вправа 2.** Встанови CPU рантайм. Запусти комірку вище і зроби скріншот виводу. Потім у Colab зміни рантайм на GPU, перезапусти всі комірки від початку. Знову зроби скріншот тієї самої комірки. Збережи обидва скріншоти — пізніше завантажиш їх на GitHub разом зі здачею лаби.

### 0.4. Монтуємо Google Drive

Сесія Colab обмежена в часі й може скинутись — усе в `/content` зникає разом
з нею. Монтуємо Drive, щоб чекпоінт моделі й логи TensorBoard пережили
перезапуск.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ARTIFACT_DIR = '/content/drive/MyDrive/ai_systems/lab1'
else:
    ARTIFACT_DIR = os.path.abspath('lab1')
os.makedirs(ARTIFACT_DIR, exist_ok=True)
RUNS_DIR = os.path.join(ARTIFACT_DIR,'runs')
os.makedirs(RUNS_DIR,exist_ok=True)
print('Артефакти зберігаються в:',ARTIFACT_DIR)

---
## 1. Ознайомлення з бібліотеками

Кілька дуже простих вправ на NumPy, pandas, PyTorch і Matplotlib — щоб мати
спільну базу перед тим, як усі чотири підуть у хід одночасно.

### 1.1. NumPy

Документація: https://numpy.org/doc/stable/

**Вправа 3.** Створи масив форми 3×4, заповнений нулями (`np.zeros`), і ще один такої самої форми, заповнений числом 7 (`np.full`). Виведи форму (`.shape`) і тип даних (`.dtype`) обох.

In [ ]:
zeros_arr = np.zeros((3,4))
sevens_arr = np.full((3,4),7)
for a in (zeros_arr,sevens_arr): print(a,a.shape,a.dtype)

**Вправа 4.** Створи масив від 0 до 20 із кроком 2 через `np.arange`, і масив із 5 рівномірно розподілених точок від 0 до 1 через `np.linspace`. Виведи обидва.

In [ ]:
arange_arr = np.arange(0,20,2)
linspace_arr = np.linspace(0,1,5)
print(arange_arr, linspace_arr)

**Вправа 5.** Створи одиничну матрицю 4×4 (`np.eye`) і випадковий масив цілих чисел форми 3×3 у діапазоні [0, 10) (`np.random.randint`). Виведи `.ndim` обох масивів.

In [ ]:
identity = np.eye(4)
random_ints = np.random.randint(0,10,size=(3,3))
for a in (identity,random_ints): print(a,a.ndim)

**Вправа 6.** Маєш масив оцінок студентів. Порахуй середнє, стандартне відхилення і z-normalized версію масиву — без циклів.

In [ ]:
scores = np.array([78,85,92,67,90,74,88,95,60,82])
mean = scores.mean()
std = scores.std()
z_scores = (scores-mean)/std
print("Середнє:",mean,"Std:",std)
print(z_scores)

**Вправа 7.** Маєш 1D масив із 12 чисел. Перетвори його на матрицю 3×4 і порахуй суму кожного рядка та кожного стовпця.

In [ ]:
arr = np.arange(1,13)
matrix = arr.reshape(3,4)
row_sums = matrix.sum(axis=1)
col_sums = matrix.sum(axis=0)
print(matrix,row_sums,col_sums)

### 1.2. pandas

Документація: https://pandas.pydata.org/docs/

**Вправа 8.** Створи таблицю студентів з оцінками. Виведи тільки тих, хто набрав більше 80 балів, відсортованих за спаданням оцінки.

In [ ]:
df = pd.DataFrame({"name":["Анна","Іван","Олег","Марія","Олена"],"score":[95,82,70,88,75]})
top = df[df.score > 80].sort_values("score",ascending=False)
print(top)

**Вправа 9.** Додай колонку `grade`: `A` якщо `score >= 90`, `B` якщо `score >= 75`, інакше `C`.

In [ ]:
df["grade"] = np.select([df.score >= 90, df.score >= 75],["A","B"],default="C")
print(df)

### 1.3. PyTorch

Документація: https://pytorch.org/docs/stable/index.html

**Вправа 10.** Створи тензор форми 2×3, заповнений нулями (`torch.zeros`), і ще один такої самої форми, заповнений числом 7 (`torch.full`). Виведи форму (`.shape`) і тип даних (`.dtype`) обох.

In [ ]:
zeros_t = torch.zeros((2,3))
sevens_t = torch.full((2,3),7)
for t in (zeros_t,sevens_t): print(t,t.shape,t.dtype)

**Вправа 11.** Створи тензор від 0 до 20 із кроком 2 через `torch.arange`, і тензор із 5 рівномірно розподілених точок від 0 до 1 через `torch.linspace`.

In [ ]:
arange_t = torch.arange(0,20,2)
linspace_t = torch.linspace(0,1,5)
print(arange_t,linspace_t)

**Вправа 12.** Створи тензор напряму зі списку — `torch.tensor([[1, 2], [3, 4]])` — і випадковий тензор форми 3×3 через `torch.rand`. Виведи `.shape`, `.dtype` і `.ndim` обох.

In [ ]:
from_list = torch.tensor([[1,2],[3,4]])
random_t = torch.rand(3,3)
for t in (from_list,random_t): print(t,t.shape,t.dtype,t.ndim)

**Вправа 13.** Створи тензор `x = 5.0` з `requires_grad=True`, порахуй `y = x**2` і перевір автоматичний градієнт проти аналітичного ($dy/dx = 2x$).

In [ ]:
n = 5.0
x = torch.tensor(n,requires_grad=True)
y = x**2
y.backward()
print("Автоматичний:",x.grad.item(),"Аналітичний:",2*n)

**Вправа 14.** Знайди $x$, який мінімізує $(x-3)^2 + 5$ — не аналітично, а градієнтним спуском: `x` як параметр з `requires_grad=True`, без жодної моделі.

In [ ]:
x = torch.tensor(0.0,requires_grad=True)
lr = 0.1
for _ in range(20):
    loss = (x-3)**2+5
    loss.backward()
    with torch.no_grad(): x -= lr*x.grad
    x.grad.zero_()
print(f"x = {x.item():.4f} (мінімум у x=3)")

### 1.4. Matplotlib

Документація: https://matplotlib.org/stable/index.html

**Вправа 15.** Побудуй графік функції $y = \sin(x)$ на проміжку $[0, 2\pi]$.

In [ ]:
xs = np.linspace(0,2*np.pi,100)
plt.plot(xs,np.sin(xs))
plt.xlabel("x"); plt.ylabel("sin(x)"); plt.grid(); plt.show()

**Вправа 16.** На одному графіку познач дві групи точок різними кольорами і додай легенду.

In [ ]:
group_a = np.random.randn(30)+2
group_b = np.random.randn(30)-2
plt.scatter(np.arange(30),group_a,color="C0",label="A")
plt.scatter(np.arange(30),group_b,color="C1",label="B")
plt.legend(); plt.show()

---
## Завдання 1. Лінійна регресія

Тренуємо базовий пайплайн на простій лінійній регресії з одним параметром.

$$y = w \cdot x + b + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 1)$$

In [ ]:
TASK1_TRUE_W = 2.0
TASK1_TRUE_B = 1.0
TASK1_N = 300

x1 = torch.empty(TASK1_N, 1).uniform_(-5, 5)
noise = 1.0 * torch.randn(TASK1_N, 1)
y1 = TASK1_TRUE_W * x1 + TASK1_TRUE_B + noise

**Вправа 17.** Перш ніж будувати модель — подивись на самі дані.

In [ ]:
plt.scatter(x1.numpy().ravel(),y1.numpy().ravel())
plt.xlabel("x1"); plt.ylabel("y1"); plt.title("Сирі дані (до тренування)"); plt.show()

**Вправа 18.** Розбий дані на train/val/test (70/15/15). Перемішай дані перед зрізом, інакше спліт вийде не випадковим.

In [ ]:
idx = torch.randperm(TASK1_N)
a,b = int(.7*TASK1_N),int(.85*TASK1_N)
x_train1,y_train1 = x1[idx[:a]],y1[idx[:a]]
x_val1,y_val1 = x1[idx[a:b]],y1[idx[a:b]]
x_test1,y_test1 = x1[idx[b:]],y1[idx[b:]]
print(f"train/val/test: {len(x_train1)}/{len(x_val1)}/{len(x_test1)}")

**Вправа 19.** Нижче — готовий цикл навчання, кожен крок підписаний окремо. Просто запусти обидві комірки й подивись, як модель наближається до `true` за 100 епох (знімки кожні 30 епох).

In [ ]:
task1_model = torch.nn.Linear(in_features=1, out_features=1)
optimizer = torch.optim.SGD(task1_model.parameters(), lr=0.05)
loss_fn = torch.nn.MSELoss()

x_line = torch.linspace(-5, 5, 100).unsqueeze(1)  # сітка для візуалізації лінії
snapshots = []  # (epoch, w, b, y_line_pred) кожні 30 епох

for epoch in range(100):
    # обнулити градієнти з попереднього кроку
    optimizer.zero_grad()
    # forward -- прогноз моделі на поточних w, b
    y_pred = task1_model(x_train1)
    # loss -- наскільки прогноз відрізняється від y_train1
    loss = loss_fn(y_pred, y_train1)
    # backward -- градієнти loss за w і b
    loss.backward()
    # крок оптимізатора -- оновити w і b
    optimizer.step()

    if epoch % 30 == 0 or epoch == 99:
        with torch.no_grad():
            y_line_pred = task1_model(x_line)
        snapshots.append((epoch, task1_model.weight.item(), task1_model.bias.item(), y_line_pred))
        print(f"epoch {epoch:4d}  loss={loss.item():.6f}  "
              f"w={task1_model.weight.item():.4f} (true {TASK1_TRUE_W})  "
              f"b={task1_model.bias.item():.4f} (true {TASK1_TRUE_B})")


In [ ]:
with torch.no_grad():
    y_line_true = TASK1_TRUE_W * x_line + TASK1_TRUE_B

fig, axes = plt.subplots(1, len(snapshots), figsize=(4 * len(snapshots), 3.5), sharey=True)
for ax, (epoch, w, b, y_line_pred) in zip(axes, snapshots):
    ax.scatter(x1.numpy(), y1.numpy(), s=8, alpha=0.3, label="дані")
    ax.plot(x_line.numpy(), y_line_true.numpy(), "g--", label="true")
    ax.plot(x_line.numpy(), y_line_pred.numpy(), "r-", label="модель")
    ax.set_title(f"epoch {epoch}\nw={w:.2f}, b={b:.2f}")
    ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout()
plt.show()


**Вправа 20.** Придумай власний напів-реальний сценарій лінійної залежності
(наприклад: кількість з'їдених пачок чіпсів на тиждень → індекс маси тіла;
години сну → продуктивність; щось своє). У markdown-комірці перед кодом
опиши сценарій: яку залежність ти
закладаєш ($w$, $b$) і чому саме такі значення видались тобі правдоподібними.
Згенеруй дані з шумом, збережи їх у CSV файл (за зразком відповідної частини лекції)
і натренуй лінійну регресію за тим самим патерном — спліт, цикл навчання, перевірка на test.
Організуй код тренування у логічні перевикористовувані функції.

**Сценарій:** години підготовки (0–8) та бали тесту. Припускаємо y = 6x + 40 + шум: 40 балів без підготовки, приблизно 6 додаткових балів за годину. Це синтетичні дані, а не реальні спостереження.

In [ ]:
set_seed()
def make_data():
    x = torch.rand(300,1)*8
    y = 6*x+40+torch.randn_like(x)*2
    return x,y

def split_data(x,y):
    ids = torch.randperm(len(x))
    a,b = int(.7*len(x)),int(.85*len(x))
    return [(x[i],y[i]) for i in (ids[:a],ids[a:b],ids[b:])]

def train_model(train,val,epochs=300):
    model = torch.nn.Linear(1,1)
    opt = torch.optim.SGD(model.parameters(),lr=.02)
    loss_fn = torch.nn.MSELoss()
    for epoch in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(train[0]),train[1])
        loss.backward()
        opt.step()
        if epoch % 50 == 0 or epoch == epochs-1:
            with torch.no_grad(): v = loss_fn(model(val[0]),val[1]).item()
            print(f'Epoch {epoch}: train={loss.item():.4f}, val={v:.4f}')
    return model

def evaluate(model,test):
    with torch.no_grad(): return torch.nn.functional.mse_loss(model(test[0]),test[1]).item()

x,y = make_data()
pd.DataFrame({'hours':x.ravel().numpy(),'score':y.ravel().numpy()}).to_csv(os.path.join(ARTIFACT_DIR,'dataset.csv'),index=False)
train,val,test = split_data(x,y)
model = train_model(train,val)
print('Test MSE:',evaluate(model,test))
print('w:',model.weight.item(),'b:',model.bias.item())
with torch.no_grad():
    line = torch.linspace(0,8,100).reshape(-1,1)
    plt.scatter(x.ravel(),y.ravel(),alpha=.4,label='Дані')
    plt.plot(line.ravel(),model(line).ravel(),label='Модель')
    plt.xlabel('Години навчання'); plt.ylabel('Бали'); plt.legend(); plt.show()

**Вправа 21.** Збережи натреновану модель на своєму гугл диску у файлі з назвою `model.pt`.
Збережений файл потрібно буде також здати разом з лабою.

In [ ]:
torch.save(model.state_dict(),os.path.join(ARTIFACT_DIR,'model.pt'))
print('Модель збережено')

**Вправа 22.** Встанови [Gradio](https://gradio.app/)
```
!pip install -q gradio
import gradio as gr
```
Завантаж свою модель, збережену в попередній вправі і створи інтерактивну аппку.

In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr
loaded_model = torch.nn.Linear(1,1)
loaded_model.load_state_dict(torch.load(os.path.join(ARTIFACT_DIR,'model.pt'),map_location='cpu',weights_only=True))
loaded_model.eval()
def predict(hours):
    with torch.no_grad(): return round(loaded_model(torch.tensor([[float(hours)]])).item(),2)
demo = gr.Interface(fn=predict,inputs=gr.Slider(0,8,step=.5,label='Години навчання'),outputs=gr.Number(label='Прогноз балів'),title='Прогноз результату тесту')
demo.launch(share=IN_COLAB,prevent_thread_lock=True)

---
### Як здавати лабу
У [форму](https://forms.gle/5qxRYshxKknBXDCz8) здати посилання на **публічний** репозиторій, який містить:
1. **Цей ноутбук з усіма виконаними вправами**. Ноутбук має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Додаткові скріншоти** по ходу виконання лаби (якими Ви доводите, що справді виконували цбю лабу).
3. Створений по ходу виконання лаби **csv файл** з даними і **pt файл** з моделлю.
4. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
    /lab1/linear_models_lab.ipynb
    /lab1/screenshots/*
    /lab1/dataset.csv
    /lab1/model.pt
```